# 菜谱数据只读审计 · 2026-10-22

本笔记只读取同目录审计汇总，不连接生产环境，不更改业务数据。全量统计 6,665 道；内容阅读样本 20 道。见调研报告中的方法、来源与限制。

In [1]:
import json
from pathlib import Path
p = Path('audit-summary.json')
if not p.exists():
    p = Path('docs/data/catalog-audit-20261022/audit-summary.json')
r = json.loads(p.read_text(encoding='utf-8'))
c = r['counts']
n = c['recipes']
print('Snapshot:', r['capturedAt'])
print('Recipes:', n)
print('SHA256:', r['snapshotSha256Uncompressed'])


Snapshot: 2026-10-22T12:51:07.598397+00:00
Recipes: 6665
SHA256: b15cbfcc5d4ddae931e2363ec2c341026f91f5f5dfebcf8257986a64a32433e4


## 总体完整性与关联

检查格式和版本绑定；通过不等于配方内容经过人工核验。

In [2]:
assert n == r['exactCounts']['food'] == r['exactCounts']['dish_quality_profile'] == 6665
assert sum(r['distributions']['type'].values()) == n
assert all(v == 0 for v in r['integrity'].values())
for key in ['invalidIngredientLines','invalidPositiveQuantities','unknownIngredientUnits','invalidImageUrls','invalidQualityProfiles','qualityRecipeVersionMismatch','qualityDishIdMismatch','qualityStatusMismatch','qualityDatasetMismatch','qualitySourceHashMismatch','qualityContentHashColumnMismatch']:
    assert c.get(key, 0) == 0, key
for kind, count in r['distributions']['type'].items():
    print(kind, count, f'{count/n:.1%}')
print('All checked associations and version bindings are consistent.')


meat 3200 48.0%
veg 2190 32.9%
soup 619 9.3%
staple 400 6.0%
dessert 256 3.8%
All checked associations and version bindings are consistent.


## 推荐标签覆盖

旧早餐标签是审核线索，不是已通过的早餐适用性结论。

In [3]:
metrics = {'Breakfast canonical':r['distributions']['tagCode'].get('BREAKFAST_ELIGIBLE',0),'Breakfast source candidates':r['breakfastRawTagCandidateCount'],'Cuisine':c['hasGroup_cuisine'],'Flavor':c['hasGroup_flavor'],'Method':c['hasGroup_method']}
for label, value in metrics.items():
    print(label, value, f'{value/n:.1%}')
assert sum(r['unmappedMethods'].values()) == n-c['hasGroup_method']
print('Unmapped methods:', r['unmappedMethods'])


Breakfast canonical 0 0.0%
Breakfast source candidates 34 0.5%
Cuisine 400 6.0%
Flavor 620 9.3%
Method 6060 90.9%
Unmapped methods: {'糖醋': 25, '煎': 358, '腌': 27, '白灼': 7, '微波': 29, '酱制': 116, '焗': 3, '汆': 18, '烙': 14, '煨': 1, '冲泡': 2, '卤': 4, '生食': 1}


## 食材、用量与来源

SOURCE_EXPLICIT 是词语匹配证据，不等同于人工审核；未知数量不转成 0。

In [4]:
assert c['rawIngredientLines'] - c['rejectedIngredientFacts'] == c['qualityIngredientFacts']
assert c['ingredientFactsWithQuantityEvidence'] == c['ingredientFactsWithQuantityValue'] == 0
assert r['distributions']['ingredientQuantityStatus']['UNKNOWN'] == c['qualityIngredientFacts']
assert sum(r['distributions']['ingredientIdentityStatus'].values()) == c['qualityIngredientFacts']
for key in ['rawIngredientLines','qualityIngredientFacts','rejectedIngredientFacts','ingredientFactsWithQuantityEvidence','recipesWithUnreviewedIngredientIdentity']:
    print(key, c[key])
print('Salt:', r['ingredients']['quantitiesForFrequentNames']['食盐'])
print('Oil:', r['ingredients']['quantitiesForFrequentNames']['食用油'])
print('Sugar:', r['ingredients']['quantitiesForFrequentNames']['白糖'])
print('Recipe review:', r['distributions']['quality_reviewStatus'])


rawIngredientLines 47161
qualityIngredientFacts 47109
rejectedIngredientFacts 52
ingredientFactsWithQuantityEvidence 0
recipesWithUnreviewedIngredientIdentity 6617
Salt: {'7克': 6430}
Oil: {'45毫升': 5486}
Sugar: {'55克': 2106}
Recipe review: {'UNREVIEWED': 6665}


## 同名、步骤、图片与时间

同名不等于内容重复。时间规则不覆盖汉字时长、小时和并行步骤；图片仅验证 URL 语法。

In [5]:
assert c['stepsTotal'] == c['numberPrefixedSteps'] == 34212
assert c['stepAndStepsIdentical'] == c['emptyStepImageArrays'] == n
print('Duplicate names:', r['duplicates']['exactName']['groups'], 'groups /',r['duplicates']['exactName']['rows'],'records')
print('Exact selected-content duplicate groups:',r['duplicates']['sameContentFingerprint']['groups'])
print('Time candidates:', c['recipesStepMinutesExceedCookMinutes'])
print('Cook time distribution:', r['distributions']['cook_minutes'])
print('Image domains:', r['distributions']['imageHost'])
print('Nutrition:', r['distributions']['kcal'],r['distributions']['quality_nutritionStatus'])


Duplicate names: 93 groups / 199 records
Exact selected-content duplicate groups: 0
Time candidates: 390
Cook time distribution: {'20': 3460, '15': 1482, '25': 986, '60': 429, '45': 252, '90': 56}
Image domains: {'cp1.douguo.net': 6580, 'cp2.douguo.net': 85}
Nutrition: {'0': 6665} {'UNKNOWN': 6665}


## 原始查询与审计边界

精确只读查询在 `queries.sql`。原始全量快照没有嵌入本笔记；汇总 SHA-256 用于关联审计产物。先保留原始资料，后续清洗写新标准版本和差异报告，不能依靠本笔记直接认证原配方。

In [6]:
print(*r['limitations'], sep=chr(10))

Full static profile; semantic inspection covers 20 recipes only.
Unknown and generated values are not verified facts.
Image URL syntax is checked; reachability, identity and usage rights are not.
390 time flags are candidates from numeric-minute literals only, not an adjudicated error rate.
Creation and profile timestamps do not establish original recipe freshness.
Related-table counts are exact COUNT(*), not information_schema cardinality estimates.
Zero counters are omitted by the source Counter; use .get(name, 0).
